# Tree-structured attractors for the ordinal structure of temporal sequences

>**Zou, Chu et al. (2023), *Learning and Processing the Ordinal Information of Temporal Sequences in Recurrent Neural Circuits*, NeurIPS 36**

Many things the brain handles — words, actions, memories — are sequences whose *content* (which items) can be separated from their *order* (which item comes first). If a neural circuit stores the order structure of a family of sequences in a content-free form, it should be able to reuse that structure when new content arrives — a "schema". This paper shows that a small recurrent network, trained to classify two-item sequences with a ramping teaching signal, spontaneously organises its dynamics into a **tree of attractors**: one attractor for the spontaneous state, one per possible first item, one per complete sequence. The tree *is* the ordinal structure, and because it lives in the recurrent weights it can be frozen and rebound to new items, which makes learning new sequences dramatically faster.

## Setup

In [ ]:
import copy
import json
import math
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

import neuralrnn  # registers all model families (import side effect)
from neuralrnn import AutoConfig, AutoModel, SupervisedObjective, Trainer, TrainingArguments
from neuralrnn.analysis import fit_pca

# ---------------------------------------------------------------- paths
NOTEBOOK_DIR = Path.cwd()
MODEL_DIR = NOTEBOOK_DIR / "models" / "tree_structure"
FIG_DIR = NOTEBOOK_DIR / "figs" / "tree_structure"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"          # 96-step rollouts of a 50-unit network: CPU is fastest
SEED = 0

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed()

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
})

In [ ]:
from __future__ import annotations

import numpy as np

# Class colours follow the paper's Fig. 2: S1 red, S2 green, S3 orange, S4 blue.
CLASS_COLORS = ["#c44e52", "#55a868", "#e5ae38", "#4c72b0"]
# Tree-node labels in the paper's notation: root; first-layer branches S_1^{1,2}
# (shared by classes 1,2) and S_1^{3,4}; second-layer leaves S_2^1..S_2^4.
NODE_LABELS = {"root": "root", "item1_0": r"$S_1^{1,2}$", "item1_1": r"$S_1^{3,4}$",
               "leaf_0": r"$S_2^1$", "leaf_1": r"$S_2^2$",
               "leaf_2": r"$S_2^3$", "leaf_3": r"$S_2^4$"}


def plot_trial_example(ax_in, ax_tg, trial: dict, idx: int = 0):
    """One augmented trial: input channels (top) and ramping targets (bottom)."""
    X, Y = trial["inputs"][idx], trial["targets"][idx]
    t1 = int(trial["pres"][idx])                # onset of the first item
    t2 = t1 + 3 + int(trial["gaps"][idx])
    for ch in range(X.shape[1]):
        ax_in.plot(X[:, ch] + 1.6 * ch, color="0.3", lw=1.0)
    for t in (t1, t2):
        ax_in.axvline(t, color="0.7", ls="--", lw=0.8)
    ax_in.set_yticks([])
    ax_in.set_ylabel("inputs")
    ax_in.set_xlabel("time step")
    for k in range(Y.shape[1]):
        ax_tg.plot(Y[:, k], color=CLASS_COLORS[k], lw=1.2, label=f"readout {k + 1}")
    ax_tg.set_ylim(-0.05, 1.1)
    ax_tg.set_ylabel("targets")
    ax_tg.set_xlabel("time step")
    ax_tg.legend(frameon=False, ncol=2)


def plot_readout_dynamics(ax, outputs: np.ndarray, gap: int, pre: int, seq_label: str,
                          legend: bool = True):
    """Readout activities for one test trial (paper Fig. 2C): coarse-to-fine."""
    T = outputs.shape[0]
    t1 = int(pre)                        # onset of the first item
    t2 = t1 + 3 + int(gap)               # onset of the second item
    for k in range(outputs.shape[1]):
        ax.plot(outputs[:, k], color=CLASS_COLORS[k], lw=1.4, label=f"readout {k + 1}")
    for t, txt in ((t1, "item 1"), (t2, "item 2")):
        ax.axvline(t, color="0.7", ls="--", lw=0.8)
        ax.text(t + 1, 1.02, txt, fontsize=7, color="0.4")
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlabel("time step")
    ax.set_ylabel("output")
    ax.set_title(f"sequence {seq_label}", fontsize=8)
    if legend:
        # below the panel: the traces rise to ~1, so any in-axes legend covers them
        ax.legend(frameon=False, ncol=4, fontsize=6.5, loc="upper center",
                  bbox_to_anchor=(0.5, -0.28), handlelength=1.2,
                  columnspacing=0.9, borderaxespad=0)


def plot_pca_variance(ax, cum_var: np.ndarray, n_show: int = 10):
    """Cumulative explained variance vs number of PCs (paper Fig. 2D, top)."""
    xs = np.arange(1, len(cum_var[:n_show]) + 1)
    ax.plot(xs, 100 * cum_var[:n_show], "o-", color="0.3", ms=3, lw=1.0)
    ax.axhline(90, color="0.6", ls=":", lw=0.8)
    ax.text(xs[-1], 91, "90%", ha="right", fontsize=7, color="0.4")
    ax.set_xlabel("# principal components")
    ax.set_ylabel("variance explained (%)")
    ax.set_ylim(0, 102)


def plot_tree_trajectories(ax, traj_pc: dict[int, np.ndarray], fp_pc: dict[str, np.ndarray],
                           seq_names, dims=(0, 1, 2)):
    """Trial-averaged trajectories in PC space with fixed points as crosses
    (paper Fig. 2E). traj_pc: class -> (T, 3); fp_pc: node name -> (3,)."""
    for k, traj in traj_pc.items():
        d = traj[:, dims]
        if len(dims) == 3:
            ax.plot(d[:, 0], d[:, 1], d[:, 2], color=CLASS_COLORS[k], lw=1.6,
                    label=seq_names[k])
        else:
            ax.plot(d[:, 0], d[:, 1], color=CLASS_COLORS[k], lw=1.6, label=seq_names[k])
    for node, p in fp_pc.items():
        d = p[list(dims)]
        if len(dims) == 3:
            ax.scatter(*d, marker="x", s=45, color="black", zorder=5)
            ax.text(*d, f" {NODE_LABELS.get(node, node)}", fontsize=7, color="0.2")
        else:
            ax.scatter(*d, marker="x", s=45, color="black", zorder=5)
            ax.text(*d, f" {NODE_LABELS.get(node, node)}", fontsize=7, color="0.2")
    if len(dims) == 3:
        ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.set_zlabel("PC 3")
    else:
        ax.set_xlabel(f"PC {dims[0] + 1}"); ax.set_ylabel(f"PC {dims[1] + 1}")
        ax.legend(frameon=False)


def plot_eigen_spectra(axes, spectra: dict[str, np.ndarray], node_titles: dict[str, str]):
    """Jacobian eigenvalues in the complex plane per tree node (paper Fig. S6)."""
    theta = np.linspace(0, 2 * np.pi, 200)
    for ax, (node, eigs) in zip(axes, spectra.items()):
        ax.plot(np.cos(theta), np.sin(theta), color="0.8", lw=0.8)
        ax.scatter(np.real(eigs), np.imag(eigs), s=6, color="0.25")
        ax.axhline(0, color="0.9", lw=0.5); ax.axvline(0, color="0.9", lw=0.5)
        ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
        ax.set_title(node_titles.get(node, node), fontsize=8)
        ax.set_xlabel("real"); ax.set_ylabel("imag")
        ax.set_aspect("equal")


def plot_transition(ax, traj_pc: dict[int, np.ndarray], fp_pc: dict[str, np.ndarray],
                    branch_node: str, leaf_nodes: tuple[str, str],
                    dirs_pc: np.ndarray, eigvals: np.ndarray, seq_names,
                    scale: float = 1.0):
    """Dominant eigenvectors at a first-layer node point to the second-layer nodes
    (paper Fig. 2F). Directions are drawn as arrows with length ∝ |eigenvalue|."""
    for k, traj in traj_pc.items():
        ax.plot(traj[:, 0], traj[:, 1], color=CLASS_COLORS[k], lw=1.2, alpha=0.7,
                label=seq_names[k])
    for node, p in fp_pc.items():
        ax.scatter(p[0], p[1], marker="x", s=50, color="black", zorder=5)
        ax.text(p[0], p[1], f" {NODE_LABELS.get(node, node)}", fontsize=7, color="0.2")
    base = fp_pc[branch_node]
    lam = np.abs(eigvals)
    lam = lam / lam.max()
    for v, l, leaf in zip(dirs_pc, lam, leaf_nodes):
        ax.arrow(base[0], base[1], scale * l * v[0], scale * l * v[1],
                 head_width=0.06 * scale, color="#8172b3", length_includes_head=True,
                 zorder=4)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.legend(frameon=False, loc="best")


def plot_transfer(ax_d, ax_a, ep_t, disc_t, ep_acc_t, acc_t, ep_acc_c, acc_c):
    """Transfer results (paper Fig. 3): discrepancy vs epoch (left) and test
    accuracy with vs without transfer (right)."""
    ax_d.plot(ep_t, disc_t, ".-", color="#c44e52", ms=3, lw=1.0)
    ax_d.set_xlabel("training epoch")
    ax_d.set_ylabel("discrepancy")
    ax_d.set_title("learned dynamics vs template", fontsize=8)
    ax_a.plot(ep_acc_t, 100 * acc_t, ".-", color="#c44e52", ms=3, lw=1.0,
              label="with transfer (frozen $W_{rec}$)")
    ax_a.plot(ep_acc_c, 100 * acc_c, ".-", color="0.35", ms=3, lw=1.0,
              label="without transfer (from scratch)")
    ax_a.set_xlabel("training epoch")
    ax_a.set_ylabel("test accuracy (%)")
    ax_a.set_ylim(0, 105)
    ax_a.legend(frameon=False, loc="lower right")

## Background: content is easy, order is the problem

Consider the four sequences that this notebook uses throughout:

$$S_1=\{a,b\},\quad S_2=\{a,c\},\quad S_3=\{b,a\},\quad S_4=\{b,c\},$$

built from three primitive items $a$, $b$, $c$. To tell the sequences apart, a network
cannot simply register which items appeared — $S_1$ and $S_3$ contain the same two items
and differ only in their *order*, as do $S_2$ and $S_4$. What matters is the **ordinal
structure**, which here has the shape of a two-layer binary tree: the first item selects
a branch ($a$ first → class 1 or 2; $b$ first → class 3 or 4), and the second item selects
a leaf.

The claim of the paper is that a recurrent circuit can store such an ordinal structure
**as a tree of attractors in its state space**, disjoint from the particular items that
fill it: the spontaneous state sits at the *root* attractor; presenting a first item
drives the state to the corresponding *branch* attractor, where it stays until the second
item pushes it to a *leaf*. Because the tree lives in the recurrent connectivity rather
than in the input pathway, it acts as a reusable **schema**: freeze the recurrent weights,
present entirely new items, and only the input and readout weights need to relearn —
which is what makes transfer to new sequences fast (the second half of this notebook).

Two design choices in the training protocol turn out to be essential, and both are
counter-intuitive enough to be worth stating up front:

1. **Variable waiting times.** The gap between the two items is random (0–30 time steps,
   filled with faint noise). The network therefore cannot solve the task by timing; the
   only way to bridge an unpredictable gap is to *hold* a state — and holding a state
   against a leak is precisely what an attractor does. Fixed gaps produce sequence
   classifiers too, but not attractor trees that survive perturbation.
2. **Ramping targets.** The readout is not asked for a one-shot answer. Before the
   sequence starts, all readouts sit at $O_{\mathrm{init}}=0.25$; after the first item,
   the two classes compatible with it rise to $O_1=0.5$ while the others decay to
   $O_{\mathrm{sile}}=0$; after the second item, the correct class rises to $O_2=1$.
   The teaching signal itself is tree-shaped, which sculpts the same shape into the
   dynamics.

## The task

**Items and sequences.** Each primitive item is a fixed 3-dimensional input vector
presented for $\Delta t = 3$ time steps:

$$a = (1,1,0),\qquad b = (1,0,1),\qquad c = (0,1,1).$$

The four sequences $S_1$–$S_4$ above tile a two-layer tree over these items.

**Trial layout and augmentation** (paper Sec. 2.2–2.3, SI Sec. F.1). One trial is:

```
[ pre period ] [ item 1 ] [ gap ] [ item 2 ] [ post period ]
   10–30 steps    3 steps   0–30     3 steps     30 steps
```

* the **gap** between items is drawn per trial from $\mathrm{Uniform}[0, 30]$ and filled
  with Gaussian white noise of strength $\sigma = 0.01$ — the temporal-warping
  augmentation that forces duration-invariant representations;
* the **pre period** is drawn from $\mathrm{Uniform}[10, 30]$ for the same reason:
  the spontaneous state must be *held* for an unpredictable time, which is what turns
  the tree's root into a genuine attractor (see the deviation note in the Summary);
* the **post period** holds the final target, forcing the network to *stay* at the leaf.

**Ramping targets**. With four readout units (one per class) and levels
$O_{\mathrm{init}}=0.25$, $O_1=0.5$, $O_2=1$, $O_{\mathrm{sile}}=0$, the target for
readout $k$ at time $t$ is piecewise constant: $O_{\mathrm{init}}$ before the first
item; $O_1$ during the gap if class $k$ is compatible with the first item, else
$O_{\mathrm{sile}}$; $O_2$ after the second item for the correct class only.

**Fresh data every batch.** Rather than pre-generating a fixed dataset, every training
batch is drawn with new random gaps, pre periods and noise — an effectively infinite
dataset. The `TreeSequenceDataset` below subclasses the framework `CustomDataset` and
overrides only `sample_batch()`, which is the single method the framework `Trainer`
calls.

In [ ]:
from __future__ import annotations

import numpy as np
import torch

from neuralrnn.data.custom_dataset import CustomDataset

# ---------------------------------------------------------------- constants
# Primitive items (paper Sec. 2.3): 3-dim continuous-time vectors.
ITEMS = {
    "a": (1.0, 1.0, 0.0),
    "b": (1.0, 0.0, 1.0),
    "c": (0.0, 1.0, 1.0),
}
SEQUENCES = (("a", "b"), ("a", "c"), ("b", "a"), ("b", "c"))
SEQ_NAMES = ("{a,b}", "{a,c}", "{b,a}", "{b,c}")
N_CLASSES = len(SEQUENCES)
INPUT_DIM = 3

# Transfer contents. The paper used three TIMIT phonemes (pcl / tcl / pau) as new
# contents: temporally extended MFCC waveforms, normalised to zero mean along the
# time dimension (SI Sec. C.1). The item identity is therefore carried by the
# *temporal pattern* of the input, not by a static vector — the network must
# integrate over the item to recognise it. We mimic this with synthetic items:
# fixed pseudo-random waveforms of TRANSFER_ITEM_DUR steps, zero-mean and unit-std
# over time in each channel (paper Fig. 3A):
#   T1 = {d, e}, T2 = {d, f}, T3 = {e, d}, T4 = {e, f}
TRANSFER_ITEM_DUR = 12


def _make_temporal_item(seed: int, dur: int = TRANSFER_ITEM_DUR) -> np.ndarray:
    """A fixed zero-mean random waveform (dur, 3), the synthetic stand-in for a
    normalised MFCC phoneme token."""
    rng = np.random.default_rng(seed)
    w = rng.standard_normal((dur, INPUT_DIM)).astype(np.float32)
    w -= w.mean(axis=0, keepdims=True)
    w /= w.std(axis=0, keepdims=True)
    return w


TRANSFER_ITEMS = {
    "d": _make_temporal_item(101),
    "e": _make_temporal_item(102),
    "f": _make_temporal_item(103),
}
TRANSFER_SEQUENCES = (("d", "e"), ("d", "f"), ("e", "d"), ("e", "f"))
TRANSFER_SEQ_NAMES = ("{d,e}", "{d,f}", "{e,d}", "{e,f}")

# Augmentation parameters (SI Sec. F.1).
ITEM_DUR = 3        # duration of one primitive item in time steps (paper: Δt = 3)
DT_MAX = 30         # max inter-item interval (paper: ΔT_max = 30)
NOISE_SIGMA = 0.01  # Gaussian noise strength inside the interval
PRE_MIN = 10        # spontaneous period before the first item is also variable,
PRE_MAX = 30        # so that the spontaneous state must be *held* — this is what
                    # makes the tree's root a genuine attractor of the dynamics
POST_STEPS = 30     # period after the second item (targets held at final values)
T_TOTAL = PRE_MAX + ITEM_DUR + DT_MAX + ITEM_DUR + POST_STEPS  # = 96

# Ramping target levels (SI Sec. A.1).
O_INIT = 0.25    # spontaneous level before the sequence
O_RAMP = 0.5     # level after the first item for the two compatible classes
O_FINAL = 1.0    # level of the correct readout after the second item
O_SILENT = 0.0   # silent level


def _item_waveform(v) -> np.ndarray:
    """Normalise an item definition to a (dur, INPUT_DIM) waveform.

    Static items (the synthetic task) are 3-dim vectors held for ITEM_DUR
    steps; temporal-pattern items (the transfer task) are already (dur, 3).
    """
    a = np.asarray(v, dtype=np.float32)
    if a.ndim == 1:
        return np.tile(a, (ITEM_DUR, 1))
    return a


def trial_length(items, sequences) -> int:
    """Total trial length for the longest admissible sequence."""
    dur = max(len(_item_waveform(items[n])) for s in sequences for n in s)
    return PRE_MAX + dur + DT_MAX + dur + POST_STEPS


# ---------------------------------------------------------------- generation
def generate_trials(n_trials, *, items=ITEMS, sequences=SEQUENCES,
                    seed=None, rng=None, fixed_gap=None, fixed_pre=None,
                    add_noise=True, labels=None):
    """Generate ``n_trials`` augmented trials of the sequence-discrimination task.

    Args:
        n_trials: number of trials.
        items: dict item name -> input, either a 3-dim vector (static item,
            presented for ITEM_DUR steps) or a (dur, 3) temporal pattern.
        sequences: tuple of (first, second) item names per class.
        seed / rng: pass a seeded ``np.random.Generator`` via ``rng`` for
            streaming generation, or ``seed`` for a reproducible fixed set.
        fixed_gap: if given, every trial uses this inter-item interval
            (used for clean visualization / probing instead of U[0, DT_MAX]).
        fixed_pre: if given, every trial uses this pre-sequence period
            (default: sampled from U[PRE_MIN, PRE_MAX], like the gap).
        add_noise: whether to fill the interval with Gaussian noise.
        labels: optional (n_trials,) array of class labels; default random.

    Returns:
        dict with inputs (n,T,3), targets (n,T,4), mask (n,T), labels (n,),
        gaps (n,), pres (n,) — all float32 numpy except labels/gaps/pres (int64).
        T is the length of the longest admissible sequence (shorter trials are
        zero-padded at the end, with targets held at their final values).
    """
    if rng is None:
        rng = np.random.default_rng(seed)
    item_mat = {k: _item_waveform(v) for k, v in items.items()}
    T = trial_length(items, sequences)

    X = np.zeros((n_trials, T, INPUT_DIM), dtype=np.float32)
    Y = np.full((n_trials, T, N_CLASSES), O_SILENT, dtype=np.float32)
    if labels is None:
        labels = rng.integers(0, N_CLASSES, n_trials)
    labels = np.asarray(labels, dtype=np.int64)
    assert len(labels) == n_trials
    gaps = (np.full(n_trials, fixed_gap, dtype=np.int64) if fixed_gap is not None
            else rng.integers(0, DT_MAX + 1, n_trials))
    pres = (np.full(n_trials, fixed_pre, dtype=np.int64) if fixed_pre is not None
            else rng.integers(PRE_MIN, PRE_MAX + 1, n_trials))

    for i in range(n_trials):
        k = int(labels[i])
        s1, s2 = sequences[k]
        w1, w2 = item_mat[s1], item_mat[s2]
        t1 = int(pres[i])                  # onset of the first item
        t2 = t1 + len(w1) + int(gaps[i])   # onset of the second item
        t3 = t2 + len(w2)                  # offset of the second item

        X[i, t1:t1 + len(w1)] = w1
        X[i, t2:t3] = w2
        if add_noise and NOISE_SIGMA > 0:
            # noise fills the interval between the two items (paper Sec. 2.3)
            X[i, t1 + len(w1):t2] += NOISE_SIGMA * rng.standard_normal(
                (t2 - t1 - len(w1), INPUT_DIM)).astype(np.float32)

        # Ramping targets (SI Sec. A.1): spontaneous -> ramp -> decision.
        compat = [j for j in range(N_CLASSES) if sequences[j][0] == s1]
        Y[i, :t1] = O_INIT
        Y[i, t1:t2] = O_SILENT
        Y[i, t1:t2, compat] = O_RAMP
        Y[i, t2:] = O_SILENT
        Y[i, t2:, k] = O_FINAL

    return {
        "inputs": X,
        "targets": Y,
        "mask": np.ones((n_trials, T), dtype=np.float32),
        "labels": labels.astype(np.int64),
        "gaps": gaps,
        "pres": pres,
    }


class TreeSequenceDataset(CustomDataset):
    """On-the-fly dataset: every batch is freshly augmented (paper Sec. 2.2).

    The framework Trainer only calls ``sample_batch()``; we override it to
    generate new random intervals and noise each step, which is equivalent to
    the paper's data augmentation with an effectively infinite dataset.
    """

    def __init__(self, batch_size: int = 16, seed: int = 0, transfer: bool = False):
        self._items = TRANSFER_ITEMS if transfer else ITEMS
        self._seqs = TRANSFER_SEQUENCES if transfer else SEQUENCES
        self._rng = np.random.default_rng(seed)
        dummy = generate_trials(batch_size, items=self._items, sequences=self._seqs,
                                rng=np.random.default_rng(seed + 1))
        super().__init__(dummy["inputs"], targets=dummy["targets"],
                         mask=dummy["mask"], mode="supervised",
                         batch_size=batch_size,
                         sequence_length=trial_length(self._items, self._seqs))
        self.batch_size = batch_size

    def sample_batch(self) -> dict[str, torch.Tensor]:
        d = generate_trials(self.batch_size, items=self._items,
                            sequences=self._seqs, rng=self._rng)
        return {
            "inputs": torch.from_numpy(d["inputs"]),
            "targets": torch.from_numpy(d["targets"]),
            "mask": torch.from_numpy(d["mask"]),
        }

In [ ]:
# one augmented example trial per class: inputs (top) and ramping targets (bottom)
trial_batch = generate_trials(4, seed=3, fixed_gap=18, fixed_pre=15,
                              labels=np.arange(4))

fig, axes = plt.subplots(4, 2, figsize=(8.5, 5.6), sharex="col")
for k in range(4):
    plot_trial_example(axes[k, 0], axes[k, 1],
                       {kk: vv[k:k + 1] for kk, vv in trial_batch.items()}, idx=0)
    axes[k, 0].set_ylabel(f"{SEQ_NAMES[k]}\ninputs", fontsize=7)
    if k == 0:
        axes[k, 0].set_title("input channels (offset for display)", fontsize=8)
        axes[k, 1].set_title("ramping targets per readout", fontsize=8)
fig.suptitle("Four example trials (one per sequence class, gap = 18 steps)", y=1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / "task_trials.png", bbox_inches="tight")
plt.show()

## The network

The paper's model (Eq. 1–2) is a vanilla continuous-time RNN with a linear readout,

$$\tau \frac{dx}{dt} = -x + W_{\mathrm{rec}}\,r + W_{\mathrm{in}}\,I(t) + I^b,
\qquad r = \tanh(x), \qquad y = W_{\mathrm{out}}\,r,$$

with $M=3$ input channels, $N=50$ recurrent units, $K=4$ readouts and time constant
$\tau=2$ (SI Sec. F.1). Discretised with the Euler method at simulation step
$\Delta t = 1$, the update becomes

$$z_{t+1} = \Big(1-\tfrac{\Delta t}{\tau}\Big)\,z_t +
\tfrac{\Delta t}{\tau}\,\big(W_{\mathrm{rec}}\tanh(z_t) + W_{\mathrm{in}} I_t + b\big),
\qquad \alpha \equiv \tfrac{\Delta t}{\tau} = \tfrac{1}{2},$$

which is exactly the framework `CTRNNModel` in `nonlinearity_mode="rate"` — the latent
variable $z$ plays the role of the total input $x$, and the firing rate is
$r = \tanh(z)$. No custom model code is needed: `AutoConfig.for_model("ctrnn", ...)`
with `dt=1.0, tau=2.0` reproduces the paper's dynamics step for step (this is verified
numerically in the test suite, `ts_tests/t2_model.py`).

Two initialisation details follow the paper: the recurrent weights are drawn from
$\mathcal{N}\big(0, (g/\sqrt{N})^2\big)$ with gain $g=0.2$, and the initial state
$h_0$ is trainable so that the network can place the trial onset at the root of the
tree. The transfer experiment later uses `reinit_input`, which redraws only the input
weights — the analogue of the paper's switch from 3-dim synthetic items to 16-channel
MFCC features, which forces a brand-new $W_{\mathrm{in}}$.

In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import numpy as np
import torch

from neuralrnn import AutoConfig, AutoModel, SupervisedObjective, Trainer, TrainingArguments


# Paper hyperparameters (SI Sec. F.1): N = 50, gain g = 0.2, batch 16, 100 epochs.
N_HIDDEN = 50
WREC_GAIN = 0.2
BATCH_SIZE = 16
N_EPOCHS = 100
STEPS_PER_EPOCH = 64          # nominal dataset of 1024 trials per epoch
MAX_STEPS = N_EPOCHS * STEPS_PER_EPOCH
LEARNING_RATE = 1e-3          # paper searches lr in [1e-4, 1e-2]
EVAL_EVERY = STEPS_PER_EPOCH  # evaluate once per nominal epoch
N_EVAL_TRIALS = 400
# Transfer runs train longer than the 100 nominal epochs of Fig. 2: the new
# contents are temporal patterns (12 steps each), and aligning the chunk states
# with the template attractors keeps improving well past epoch 100.
TRANSFER_EPOCHS = 300
TRANSFER_MAX_STEPS = TRANSFER_EPOCHS * STEPS_PER_EPOCH


def reinit_input(model: torch.nn.Module, seed: int = 11,
                 gain: float = WREC_GAIN) -> torch.nn.Module:
    """Fresh random input weights for a transfer run.

    In the paper the transfer task changes the input dimension (3-dim synthetic
    items -> 16-dim MFCC), so W_in is necessarily new. Our synthetic transfer
    items keep 3 channels, so we re-initialise W_in explicitly: the readout
    W_out is inherited from the template, and the fresh W_in must learn to
    drive the frozen dynamics into the template's basins.
    """
    gen = torch.Generator().manual_seed(seed)
    w = model.input2h.weight
    with torch.no_grad():
        w.copy_(gain / math.sqrt(w.shape[1]) * torch.randn(*w.shape, generator=gen))
    return model


def build_model(seed: int = 0) -> torch.nn.Module:
    """CTRNN matching the paper: M=3 inputs, N=50 units, K=4 readouts, tau=2.

    W_rec is drawn from N(0, (g/sqrt(N))^2) with g = 0.2 (SI Sec. F.1); all other
    weights keep the framework default initialisation.
    """
    cfg = AutoConfig.for_model(
        "ctrnn", input_dim=3, latent_dim=N_HIDDEN, output_dim=4,
        activation="tanh", nonlinearity_mode="rate", dt=1.0, tau=2.0,
        trainable_h0=True,   # lets the network start each trial at the root attractor
    )
    model = AutoModel.from_config(cfg)
    gen = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        model.h2h.weight.copy_(
            WREC_GAIN / math.sqrt(N_HIDDEN) * torch.randn(N_HIDDEN, N_HIDDEN, generator=gen))
    return model


def _accuracy(model, eval_data: dict) -> float:
    """Fraction of trials whose argmax readout at the final step is correct."""
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        out = model(torch.from_numpy(eval_data["inputs"]).to(device))
    pred = out.outputs[:, -1, :].argmax(dim=-1).cpu().numpy()
    model.train()
    return float((pred == eval_data["labels"]).mean())


def make_eval_fn(eval_data: dict, probe: dict | None = None,
                 template_fps: dict | None = None):
    """eval_fn for the Trainer: test accuracy, plus the template discrepancy
    (SI Sec. C.2) when a probe set and template attractors are given."""

    def eval_fn(model):
        logs = {"test_acc": _accuracy(model, eval_data)}
        if probe is not None and template_fps is not None:
            logs["discrepancy"] = template_discrepancy(model, probe, template_fps)
        return logs
    return eval_fn


def train_or_load(name: str, model_dir: Path, *, dataset, eval_data: dict,
                  model: torch.nn.Module | None = None, seed: int = 0,
                  probe: dict | None = None, template_fps: dict | None = None,
                  freeze_recurrent: bool = False, max_steps: int = MAX_STEPS,
                  lr_scheduler: str | None = None, device: str = "cpu"):
    """Load-first training: if ``name`` under ``model_dir`` holds a completed run
    (config.json + run_meta.json), load and return (model, history); otherwise
    train, save, and return. A fresh Trainer is always built *after* any
    freezing, because the optimizer collects parameters at construction.

    Transfer protocol (paper Sec. 3.1): only the feedforward weights W_in and
    the readout W_out are optimized. Everything that shapes the intrinsic
    dynamics stays frozen — W_rec, the background bias I^b, and the initial
    state h0 — so the attractor template itself cannot move.
    """
    out_dir = Path(model_dir) / name
    meta_path = out_dir / "run_meta.json"
    hist_path = out_dir / "history.json"

    if meta_path.exists() and (out_dir / "config.json").exists():
        model = AutoModel.from_pretrained(str(out_dir), map_location=device)
        history = json.loads(hist_path.read_text())
        return model, history

    out_dir.mkdir(parents=True, exist_ok=True)
    if model is None:
        model = build_model(seed=seed)
    # analysis tools (e.g. the fixed-point finder) set requires_grad=False on
    # every parameter of the model they inspect; re-enable training before
    # applying the run's own freeze spec
    for p in model.parameters():
        p.requires_grad_(True)
    if freeze_recurrent:
        # freeze the full intrinsic dynamics: W_rec, h0, and the bias I^b
        # (input2h.bias plays the role of I^b in the paper's Eq. 1)
        frozen = model.freeze_parameters(groups=["recurrent", "h0"],
                                         patterns=[r"input2h\.bias"])
        print(f"[{name}] frozen: {frozen}")

    args = TrainingArguments(
        learning_rate=LEARNING_RATE, max_steps=max_steps, optimizer="adam",
        grad_clip_norm=1.0, log_every=STEPS_PER_EPOCH, eval_every=EVAL_EVERY,
        lr_scheduler=lr_scheduler,
        output_dir=str(out_dir), device=device, seed=seed,
    )
    trainer = Trainer(model, dataset, SupervisedObjective(task_type="regression"),
                      args, eval_fn=make_eval_fn(eval_data, probe, template_fps))
    t0 = time.time()
    trainer.train()
    wall_time = time.time() - t0

    model.save_pretrained(str(out_dir))
    history = {"train": trainer.history, "eval": trainer._eval_history,
               "wall_time_s": wall_time,
               "seed": seed, "freeze_recurrent": freeze_recurrent}
    hist_path.write_text(json.dumps(history))
    meta_path.write_text(json.dumps(  # completion flag, written last
        {"name": name, "seed": seed, "max_steps": max_steps,
         "freeze_recurrent": freeze_recurrent}, indent=2))
    return model, history


def eval_history(history: dict):
    """Extract (epochs, test_acc) and (epochs, discrepancy) from a run history."""
    evals = history["eval"]
    ep = [h["step"] / STEPS_PER_EPOCH for h in evals]
    acc = [h["test_acc"] for h in evals]
    disc = [h.get("discrepancy") for h in evals]
    return np.array(ep), np.array(acc), np.array(disc, dtype=float)

In [ ]:
from __future__ import annotations

import numpy as np
import torch

from neuralrnn.analysis import fit_pca
from neuralrnn.analysis.fixed_points import NumericFixedPointFinder
from neuralrnn.analysis.linearization import linearize


TREE_NODES = ["root", "item1_0", "item1_1", "leaf_0", "leaf_1", "leaf_2", "leaf_3"]


def run_model(model, inputs: np.ndarray):
    """Roll out the model on inputs (n,T,3); returns (states (n,T,N), outputs (n,T,4))."""
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        out = model(torch.from_numpy(inputs).to(device))
    return out.states.cpu().numpy(), out.outputs.cpu().numpy()


def chunk_end_states(model, probe: dict, n_avg: int = 5) -> dict[str, np.ndarray]:
    """Tree-node state vectors: average latent state at the end of each chunk.

    Following SI Sec. C.2, an item and its following interval form a temporal
    chunk; the state vector of a chunk is the average over the last ``n_avg``
    steps before the next item (or the trial end). Returns a dict keyed by
    node name: "root", "item1_0"/"item1_1" (state after first item 0/1),
    "leaf_0".."leaf_3" (state at the end of each class's sequence).
    """
    states, _ = run_model(model, probe["inputs"])
    labels = probe["labels"]
    gaps = probe["gaps"]
    pres = probe["pres"]
    T = states.shape[1]

    # root: end of the spontaneous period (per trial, pre-period varies)
    root = np.stack([states[i, pres[i] - n_avg:pres[i]].mean(axis=0)
                     for i in range(states.shape[0])]).mean(axis=0)

    item1 = {0: [], 1: []}
    leaf = {k: [] for k in range(N_CLASSES)}
    for i in range(states.shape[0]):
        s1 = probe["sequences"][labels[i]][0]
        t2 = int(pres[i]) + probe["dur1"][s1] + int(gaps[i])  # second-item onset
        first = 0 if probe["sequences"][labels[i]][0] == probe["order"][0] else 1
        item1[first].append(states[i, t2 - n_avg:t2].mean(axis=0))
        leaf[int(labels[i])].append(states[i, T - n_avg:].mean(axis=0))

    out = {"root": root}
    for j in (0, 1):
        out[f"item1_{j}"] = np.mean(item1[j], axis=0)
    for k in range(N_CLASSES):
        out[f"leaf_{k}"] = np.mean(leaf[k], axis=0)
    return out


def make_probe(n_per_class: int = 8, *, sequences, items, gap: int = 15,
               pre: int = 20, seed: int = 99):
    """Fixed, noise-free, class-balanced probe set for chunk states / discrepancy."""
    labels = np.tile(np.arange(N_CLASSES), n_per_class)
    probe = generate_trials(len(labels), items=items, sequences=sequences,
                            seed=seed, fixed_gap=gap, fixed_pre=pre,
                            add_noise=False, labels=labels)
    probe["sequences"] = sequences
    probe["order"] = tuple(sorted({s[0] for s in sequences}))  # first-layer items
    probe["dur1"] = {name: len(w) for name, w in
                     ((n, _item_waveform(items[n])) for s in sequences for n in s)}
    return probe


def find_tree_fixed_points(model, init_states: np.ndarray, n_candidates: int = 256,
                           seed: int = 0, speed_tol: float = 1e-5,
                           n_iters: int = 5000):
    """Find fixed points at zero input (paper SI Sec. B.2: q = 1/2 ||F(r) - r||^2).

    Candidates are initialised from states visited on test trajectories, as in
    the paper. Returns the FixedPointSet.
    """
    rng = np.random.default_rng(seed)
    flat = init_states.reshape(-1, init_states.shape[-1])
    idx = rng.choice(len(flat), size=min(n_candidates, len(flat)), replace=False)
    init = torch.from_numpy(flat[idx].astype(np.float32))
    finder = NumericFixedPointFinder(n_iters=n_iters, lr=1e-2, speed_tol=speed_tol,
                                     dedup_tol=1e-2)
    return finder.find(model, task_input=torch.zeros(model.config.input_dim),
                       init_states=init)


def match_fixed_points(fps, template_chunks: dict[str, np.ndarray]):
    """Assign each fixed point to the nearest tree node (by Euclidean distance).

    Returns dict node name -> FixedPoint. Nodes with no fixed point are absent.
    """
    remaining = list(fps.points)
    assignment = {}
    # greedy nearest-neighbour matching, closest pairs first
    pairs = []
    for node, vec in template_chunks.items():
        for p in remaining:
            pairs.append((np.linalg.norm(p.z - vec), node, p))
    pairs.sort(key=lambda t: t[0])
    used_nodes, used_pts = set(), set()
    for dist, node, p in pairs:
        if node in used_nodes or id(p) in used_pts:
            continue
        assignment[node] = p
        used_nodes.add(node)
        used_pts.add(id(p))
    return assignment


def node_spectrum(model, fp_z: np.ndarray):
    """Jacobian eigenvalues of the discrete map at a fixed point (SI Sec. B.2)."""
    device = next(model.parameters()).device
    lin = linearize(model, torch.as_tensor(fp_z, dtype=torch.float32, device=device),
                    task_input=torch.zeros(model.config.input_dim, device=device))
    return lin


def transition_directions(model, fp_z: np.ndarray, pca, n_dirs: int = 2):
    """Dominant unstable eigenvectors at a node, projected onto the PCs (Fig. 2F).

    Returns (vectors (n_dirs, n_pc), eigenvalues (n_dirs,)): the right
    eigenvectors with the largest |eigenvalue|, projected with ``pca.transform``
    applied to displacement vectors (i.e. without subtracting the mean).
    """
    device = next(model.parameters()).device
    lin = linearize(model, torch.as_tensor(fp_z, dtype=torch.float32, device=device),
                    task_input=torch.zeros(model.config.input_dim, device=device))
    eigvals, eigvecs = lin.eigenvalues, lin.eigenvectors
    order = np.argsort(-np.abs(eigvals))[:n_dirs]
    vecs = np.real(eigvecs[:, order])          # (N, n_dirs)
    proj = pca.components @ vecs               # project onto PC axes (n_pc, n_dirs)
    return proj.T, eigvals[order]


def template_discrepancy(model, probe: dict, template_fps: dict[str, np.ndarray]) -> float:
    """Mean Euclidean distance between chunk-end states and the template attractors.

    SI Sec. C.2: the final discrepancy is averaged over the seven attractor
    states of the tree (root, two branch nodes, four leaves).
    """
    chunks = chunk_end_states(model, probe)
    dists = [np.linalg.norm(chunks[node] - template_fps[node]) for node in TREE_NODES]
    return float(np.mean(dists))

## Training

The loss is the mean squared error between the readout and the ramping targets,
averaged over time and the batch (SI Eq. 3):

$$\mathcal{L} = \frac{1}{B}\sum_{i=1}^{B}\frac{1}{T}\sum_{t=1}^{T}
\big\|\,y^{(i)}_t - \hat{y}^{(i)}_t\,\big\|^2 .$$

We train with Adam (learning rate $10^{-3}$, within the paper's search range
$[10^{-4}, 10^{-2}]$) for 100 nominal epochs of 64 batches of 16 trials
(6,400 gradient steps). Because every batch is freshly augmented, the network never
sees the same temporal warping twice.

All runs in this notebook are **load-first**: `train_or_load` returns the saved
checkpoint if a completed run exists under `models/tree_structure/`, and only trains
when it is missing (a full run takes about three minutes on CPU).

In [ ]:
# --- the Fig. 2 model: everything trained on the synthetic tree task ---
train_ds = TreeSequenceDataset(batch_size=BATCH_SIZE, seed=0)
eval_data = generate_trials(N_EVAL_TRIALS, seed=123)   # fixed held-out test set

t0 = time.time()
model, hist_scratch = train_or_load(
    "scratch", MODEL_DIR, dataset=train_ds, eval_data=eval_data,
    seed=SEED, max_steps=MAX_STEPS, device=DEVICE)
acc_scratch = hist_scratch["eval"][-1]["test_acc"]
print("scratch model ready (wall time %.0fs, final test accuracy %.3f)"
      % (time.time() - t0, acc_scratch))

In [ ]:
# training curves: masked MSE (left) and held-out accuracy (right)
ep_eval, acc_eval, _ = eval_history(hist_scratch)
train_steps = np.array([h["step"] for h in hist_scratch["train"]])
train_loss = np.array([h["loss"] for h in hist_scratch["train"]])

fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.4))
axes[0].plot(train_steps / STEPS_PER_EPOCH, train_loss, lw=0.5, color="0.3")
axes[0].set(xlabel="epoch", ylabel="masked MSE", yscale="log")
axes[1].plot(ep_eval, 100 * acc_eval, ".-", color="#c44e52", ms=3, lw=1.0)
axes[1].axhline(25, color="0.6", ls=":", lw=0.8)
axes[1].text(99, 27, "chance (4 classes)", ha="right", fontsize=7, color="0.4")
axes[1].set(xlabel="epoch", ylabel="test accuracy (%)", ylim=(0, 105))
fig.suptitle("Training on the tree task (100 epochs, fresh augmentation each batch)")
fig.tight_layout()
fig.savefig(FIG_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

## The readout disentangles order from content, coarse to fine

The paper's Fig. 2C shows the signature behaviour: after the **first** item, the two
readouts compatible with it rise together to about 0.5 — the network reports the
*branch* (the coarse, order-only information) without yet committing to a class. Only
when the **second** item arrives does the correct readout climb to 1 while its sibling
collapses — the fine discrimination. Order is read out before content, because the
dynamics literally visit the branch node on the way to the leaf.

Below we plot the four readouts over a full trial for one test sequence of each class.
The dashed lines mark the two item onsets.

In [ ]:
# fixed, noise-free probe trials: n_per_class per class, identical gap/pre so that
# chunk states and trajectories are comparable across classes
probe = make_probe(n_per_class=8, sequences=SEQUENCES, items=ITEMS)
states_probe, outputs_probe = run_model(model, probe["inputs"])

fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.3), sharey=True)
for k, ax in enumerate(axes):
    i = int(np.flatnonzero(probe["labels"] == k)[0])  # first probe trial of class k
    plot_readout_dynamics(ax, outputs_probe[i], gap=int(probe["gaps"][i]),
                          pre=int(probe["pres"][i]), seq_label=SEQ_NAMES[k],
                          legend=False)
    if k:
        ax.set_ylabel("")
# one shared legend below the panels (readout k tracks class k)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=4, frameon=False, loc="lower center", fontsize=7.5)
fig.subplots_adjust(bottom=0.26)
fig.suptitle("Readout dynamics on four probe sequences (paper Fig. 2C)", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "readout_dynamics.png", bbox_inches="tight")
plt.show()

## The dynamics grow a tree of attractors

Now the heart of the paper (Fig. 2D–F). Three analyses, all on the framework's stock
tools:

**Dimensionality.** We collect the latent states over 400 held-out test trials and run
PCA (`neuralrnn.analysis.fit_pca`). The paper finds that three components capture ~90%
of the variance — the trajectories live on a low-dimensional sheet, which is what makes
the tree visible at all.

**Fixed points.** With no input, the network's state should rest exactly at the tree's
nodes. Following SI Sec. B.2 we minimise the speed
$q(z) = \tfrac{1}{2}\|F(z, I{=}0) - z\|^2$ of the discrete map, starting from states the
network actually visits on test trials (`NumericFixedPointFinder`), and keep points with
$q$ below tolerance. The search should recover **seven** points — the root, two branch
nodes, four leaves — and matching them to the tree is done by nearest-neighbour
assignment against the *chunk states*: for each probe trial we average the last five
steps before each item onset (SI Sec. C.2).

**Stability and transitions.** Linearising the map at each node
(`neuralrnn.analysis.linearize`) gives the Jacobian eigenvalues. Leaves should be stable
(all $|\lambda| < 1$); the branch nodes sit slightly unstable — and their *most unstable
eigenvectors should point at their own children* (Fig. 2F): the direction along which the
state can most easily leave the branch is the direction the next item will push it.

In [ ]:
# --- PCA over held-out test responses (paper: ~90% variance in the top 3 PCs) ---
states_test, _ = run_model(model, eval_data["inputs"])
pca = fit_pca(states_test.reshape(-1, states_test.shape[-1]), n_components=10)
cum_var = np.cumsum(pca.explained_variance_ratio)
print("cumulative variance explained by the first PCs:",
      np.round(cum_var[:5], 3))

# --- fixed points at zero input, matched to the seven tree nodes ---
fps = find_tree_fixed_points(model, states_test, n_candidates=256, n_iters=5000)
chunks = chunk_end_states(model, probe)          # chunk-end state per tree node
assignment = match_fixed_points(fps, chunks)     # node name -> nearest fixed point
print(f"found {len(fps)} fixed points; matched tree nodes: {sorted(assignment)}")
for node in TREE_NODES:
    p = assignment[node]
    print(f"  {NODE_LABELS[node]:>10s}  speed q = {p.speed:.2e}")

In [ ]:
# figure: PCA variance (left) and trial-averaged tree trajectories in PC space
# with the seven fixed points marked (right) — paper Fig. 2D (top) and 2E
traj_pc = {}
for k in range(N_CLASSES):
    sel = probe["labels"] == k
    traj_pc[k] = pca.transform(states_probe[sel].mean(axis=0))  # average over trials
fp_pc = {node: pca.transform(p.z[None])[0] for node, p in assignment.items()}

fig = plt.figure(figsize=(9.5, 3.4))
ax0 = fig.add_subplot(1, 2, 1)
plot_pca_variance(ax0, cum_var)
ax0.set_title("variance explained (paper Fig. 2D, top)", fontsize=8)
ax1 = fig.add_subplot(1, 2, 2, projection="3d")
plot_tree_trajectories(ax1, traj_pc, fp_pc, SEQ_NAMES)
ax1.set_title("tree trajectories and fixed points (paper Fig. 2E)", fontsize=8)
ax1.legend(frameon=False, fontsize=7, loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "tree_pca.png", bbox_inches="tight")
plt.show()

In [ ]:
# stability: Jacobian eigenvalue spectrum at every tree node (paper Fig. S6)
spectra = {node: node_spectrum(model, assignment[node].z).eigenvalues
           for node in TREE_NODES}

fig, axes = plt.subplots(2, 4, figsize=(9.5, 4.6))
plot_eigen_spectra(axes.ravel(), spectra, NODE_LABELS)
axes.ravel()[-1].axis("off")
fig.suptitle("Jacobian eigenvalues at the seven nodes (unit circle = stability boundary)")
fig.tight_layout()
fig.savefig(FIG_DIR / "node_spectra.png", bbox_inches="tight")
plt.show()

for node in TREE_NODES:
    lam = np.abs(spectra[node]).max()
    print(f"  {NODE_LABELS[node]:>10s}  max |eigenvalue| = {lam:.3f}  "
          f"({'stable' if lam < 1 else 'slow saddle'})")

## How the state moves between nodes

The branch nodes are *slow saddles*: almost every direction is stable, but one or two
eigenvalues peek above 1. Those weakly unstable directions are the doors out of the
node — and the paper's Fig. 2F shows they open exactly onto the node's children. We
compute the two dominant eigenvectors at the first-layer node $S_1^{1,2}$ (the branch
shared by classes $\{a,b\}$ and $\{a,c\}$), project them into the PCA plane, and draw
them as arrows whose length is proportional to the eigenvalue magnitude. They should
point from $S_1^{1,2}$ toward the leaves $S_2^1$ and $S_2^2$ — and not toward the other
branch's leaves.

In [ ]:
# transition directions at the branch node S_1^{1,2} (paper Fig. 2F)
branch, leaves = "item1_0", ("leaf_0", "leaf_1")
dirs_pc, eigvals = transition_directions(model, assignment[branch].z, pca, n_dirs=2)
fp_pc_2d = {node: v[:2] for node, v in fp_pc.items()}
traj_pc_2d = {k: v[:, :2] for k, v in traj_pc.items()}

fig, ax = plt.subplots(figsize=(4.6, 4.0))
plot_transition(ax, traj_pc_2d, fp_pc_2d, branch, leaves, dirs_pc, eigvals,
                SEQ_NAMES, scale=3.0)
ax.set_title("dominant eigenvectors at $S_1^{1,2}$ point to its children (Fig. 2F)")
fig.tight_layout()
fig.savefig(FIG_DIR / "transition_directions.png", bbox_inches="tight")
plt.show()

# quantitative check: cosine between each eigenvector and the displacement to the leaf
base = fp_pc_2d[branch]
for v, leaf in zip(dirs_pc, leaves):
    target = fp_pc_2d[leaf] - base
    cos = float(np.dot(v[:2], target) / (np.linalg.norm(v[:2]) * np.linalg.norm(target)))
    print(f"cos(eigenvector, {NODE_LABELS[branch]} -> {NODE_LABELS[leaf]}) = {cos:.3f}")

## Transfer: new content, old tree

Now the payoff (paper Fig. 3). If the tree really is a content-free schema, we should be
able to **bind new items onto it** without touching the dynamics that store it.

**New contents.** The paper switched from synthetic items to spoken phonemes (*pcl*,
*tcl*, *pau* from TIMIT), turned into MFCC features and normalised to zero mean along
time — so each item is a *temporal pattern*, not a static vector. We do the same with
synthetic stand-ins: each new item $d, e, f$ is a fixed pseudo-random waveform of 12
steps, zero-mean and unit-variance over time in each of the 3 input channels. The four
new sequences $\{d,e\}, \{d,f\}, \{e,d\}, \{e,f\}$ have exactly the tree of the old task.

**The transfer protocol** (paper Sec. 3.1). Everything that shapes the intrinsic
dynamics is frozen — $W_{\mathrm{rec}}$, the background bias $I^b$, and the initial
state $h_0$ — so the attractor template cannot move. $W_{\mathrm{in}}$ is freshly
randomised (the paper's switch to 16-channel MFCC forced a new input matrix), and only
$W_{\mathrm{in}}$ and $W_{\mathrm{out}}$ are optimised, with the same ramping targets.

**Measuring template reuse** (SI Sec. C.2). An item plus its following interval is a
temporal *chunk*; the chunk's state vector is the average over its last five steps. If
the new items are being allocated onto the old tree, each chunk state should land on the
corresponding template attractor. The **discrepancy** is the mean Euclidean distance
between the seven chunk states and the seven template fixed points. The **control** is
the same new task trained from a random initialisation with nothing frozen.

In [ ]:
# --- template: the scratch model's seven attractors ---
template_fps = {node: p.z for node, p in assignment.items()}

# the intrinsic floor: even the template's own chunk states sit slightly off the
# fixed points (the branch nodes are slow saddles, approached along a slow manifold)
floor = {node: float(np.linalg.norm(chunks[node] - template_fps[node]))
         for node in TREE_NODES}
print("discrepancy floor (template model against its own attractors):",
      round(float(np.mean(list(floor.values()))), 3))

# --- transfer run: frozen dynamics, fresh input weights, new temporal-pattern items ---
transfer_ds = TreeSequenceDataset(batch_size=BATCH_SIZE, seed=1, transfer=True)
eval_transfer = generate_trials(N_EVAL_TRIALS, items=TRANSFER_ITEMS,
                                sequences=TRANSFER_SEQUENCES, seed=321)
probe_transfer = make_probe(n_per_class=8, sequences=TRANSFER_SEQUENCES,
                            items=TRANSFER_ITEMS)

t0 = time.time()
model_transfer = reinit_input(copy.deepcopy(model))
model_transfer, hist_transfer = train_or_load(
    "transfer", MODEL_DIR, dataset=transfer_ds, eval_data=eval_transfer,
    model=model_transfer, seed=1, freeze_recurrent=True,
    max_steps=TRANSFER_MAX_STEPS, device=DEVICE,
    probe=probe_transfer, template_fps=template_fps)
print(f"transfer model ready (wall time {time.time() - t0:.0f}s)")

# --- control: the same new task, trained from scratch with nothing frozen ---
control_ds = TreeSequenceDataset(batch_size=BATCH_SIZE, seed=1, transfer=True)
model_control, hist_control = train_or_load(
    "control", MODEL_DIR, dataset=control_ds, eval_data=eval_transfer,
    seed=7, freeze_recurrent=False, max_steps=TRANSFER_MAX_STEPS, device=DEVICE)

In [ ]:
# transfer results (paper Fig. 3A right, Fig. 3B):
# discrepancy against the template, and test accuracy with vs without transfer
ep_t, acc_t, disc_t = eval_history(hist_transfer)
ep_c, acc_c, _ = eval_history(hist_control)

fig, axes = plt.subplots(1, 2, figsize=(7.8, 2.6))
plot_transfer(axes[0], axes[1], ep_t, disc_t, ep_t, acc_t, ep_c, acc_c)
axes[0].axhline(float(np.mean(list(floor.values()))), color="0.6", ls=":", lw=0.8)
axes[0].text(ep_t[-1], float(np.mean(list(floor.values()))) + 0.15,
             "intrinsic floor", ha="right", fontsize=7, color="0.4")
# the acceleration is an early-phase effect — zoom the accuracy panel on it
axes[1].set_xlim(0, 60)
axes[1].set_title("learning is accelerated (first 60 epochs)", fontsize=8)
fig.suptitle("Binding new contents onto the frozen tree (paper Fig. 3)")
fig.tight_layout()
fig.savefig(FIG_DIR / "transfer.png", bbox_inches="tight")
plt.show()

def epochs_to(acc, thr):
    hit = np.flatnonzero(acc >= thr)
    return int(hit[0]) + 1 if len(hit) else len(acc) + 1
for thr in (0.5, 0.9):
    print(f"epochs to {int(thr * 100)}% test accuracy: transfer {epochs_to(acc_t, thr)}, "
          f"control {epochs_to(acc_c, thr)}")

In [ ]:
# the transferred trajectories, drawn in the template's PC space (paper Fig. 3A left):
# new contents should walk the old tree. Temporal-pattern inputs make the raw
# trajectories jagged during item presentation, so we smooth the trial-averaged
# trajectories with a centred 9-step moving average for display.
def smooth_time(x, w=9):
    """Centred moving average along the time axis of a (T, N) array."""
    kern = np.ones(w) / w
    return np.apply_along_axis(lambda v: np.convolve(v, kern, mode="same"), 0, x)

states_transfer, _ = run_model(model_transfer, probe_transfer["inputs"])
traj_transfer = {}
for k in range(N_CLASSES):
    sel = probe_transfer["labels"] == k
    mean_traj = smooth_time(states_transfer[sel].mean(axis=0))
    traj_transfer[k] = pca.transform(mean_traj)  # template PCs

fig = plt.figure(figsize=(5.2, 4.4))
ax = fig.add_subplot(111, projection="3d")
plot_tree_trajectories(ax, traj_transfer, fp_pc, TRANSFER_SEQ_NAMES)
ax.set_title("transfer trajectories in the template PC space")
ax.legend(frameon=False, fontsize=7, loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "transfer_tree_pca.png", bbox_inches="tight")
plt.show()

## Summary

**What was reproduced.** Training a 50-unit rate network on the four-sequence task with
variable gaps and ramping targets produces (i) perfect, duration-invariant sequence
classification; (ii) coarse-to-fine readout — the branch is reported before the class
(Fig. 2C); (iii) a low-dimensional state space whose seven fixed/slow points are exactly
the nodes of the ordinal tree (Fig. 2D–E): four stable leaves, two slow-saddle branches
whose dominant unstable eigenvectors point at their own children (Fig. 2F), and a root
that is held for a variable spontaneous period. Freezing the intrinsic dynamics and
rebinding new temporal-pattern items transfers the schema: test accuracy climbs much
faster than a from-scratch control, and the chunk states move monotonically toward the
template attractors (Fig. 3).

**Deviations from the paper** (all deliberate):

1. **Variable pre-sequence period.** We randomise the pre-item interval
   ($\mathrm{Uniform}[10,30]$) exactly like the gap. With a fixed pre period the
   spontaneous state is only a transient: at zero input every trajectory eventually
   slides down to a leaf, and no root attractor forms. Making the wait unpredictable
   forces the network to *hold* the root, which completes the attractor tree. (The
   paper's Fig. S10 shows the tree also emerges with fixed intervals on clean data;
   we simply make the logic uniform across all waiting periods.)
2. **Synthetic transfer contents.** Instead of TIMIT phonemes as 16-channel MFCC
   features, the new items are fixed zero-mean random waveforms (12 steps, 3 channels).
   This keeps the key property — item identity lives in a *temporal pattern* that must
   be integrated, so static shortcuts cannot bypass the tree — without external data.
3. **Transfer budget.** Transfer trains for 300 nominal epochs (paper: 100), because the
   chunk-state alignment keeps improving long after accuracy saturates; the discrepancy
   drops about three-fold and approaches the intrinsic floor set by the slow-saddle
   branch nodes (the floor line in the discrepancy panel).
4. **Control.** Our "without transfer" baseline trains the new task from a random
   initialisation (Fig. 3B's comparison). The paper's additional cross-entropy-pretrained
   control (Fig. S7B) is not reproduced here.
5. **Epoch definition.** One nominal epoch = 64 batches of 16 freshly augmented trials
   (1,024 trials); the paper generates 10,000 augmented trials per dataset but does not
   specify the batches per epoch.

**Framework mapping.** Model: `AutoConfig.for_model("ctrnn", nonlinearity_mode="rate",
activation="tanh", dt=1.0, tau=2.0, trainable_h0=True)` — exactly the paper's Eq. 1–2.
Data: `TreeSequenceDataset`, a `CustomDataset` subclass whose `sample_batch()` emits
freshly augmented trials. Training: framework `Trainer` + `SupervisedObjective`
(masked MSE) + `TrainingArguments`; freezing via `model.freeze_parameters(groups=
["recurrent", "h0"], patterns=[...])`. Analysis: `fit_pca`, `NumericFixedPointFinder`,
`linearize` — no analysis code specific to this paper.

### References

1. Zou, Q., Chu, T. et al. (2023). *Learning and Processing the Ordinal Information of
   Temporal Sequences in Recurrent Neural Circuits.* NeurIPS 36.
2. Sussillo, D. & Barak, O. (2013). *Opening the black box: low-dimensional dynamics in
   high-dimensional recurrent neural networks.* Neural Computation 25, 626–649
   (the fixed-point-finding objective used above).

*Notebook generated from the tested modules in `ts_code/` by
`ts_tests/build_notebook_ts.py`; figures are written to `figs/tree_structure/`.*